# MAP + Laplace against NUTS

Two posteriors of the seven ground truths, built from the same noise draw: the Laplace
Gaussian at the MAP and the NUTS samples. If their bands agree, the Gaussian
approximation is adequate.

```
python src/plasma_state_estimation_for_diagnostic_placement_optimization/results_plots_tables/results/plasma_state_estimation/mcmc_benchmark/run_study.py --arms map_laplace --workers 8
python src/plasma_state_estimation_for_diagnostic_placement_optimization/results_plots_tables/results/plasma_state_estimation/mcmc_benchmark/run_study.py --arms mcmc_nuts --workers 8
```

The runner stores the runs and writes `tables/`; this notebook writes `figures/`.

In [ ]:
import json
import pathlib

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from plasma_state_estimation_for_diagnostic_placement_optimization.bayesian_inference import (  # noqa: E501
    desc_truth_types,
    profile_parametrization_types,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.mhd_equilibrium_forward_models import (  # noqa: E501
    dataset_utils,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.results_plots_tables.results.plasma_state_estimation.prior_likelihood_posterior import (  # noqa: E501
    run_study as prior_study,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.results_plots_tables.utils import (  # noqa: E501
    inference_metrics_utils,
    inference_store_utils,
    pubfig,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.results_plots_tables.utils import (  # noqa: E501
    plots_plasma_state_estimation as plots,
)

pubfig.apply_style()
pd.set_option("display.max_colwidth", None)

PACKAGE_DIRECTORY = pathlib.Path(plots.__file__).parents[2]
NOTEBOOK_DIRECTORY = (
    PACKAGE_DIRECTORY / "results_plots_tables" / "results" / "plasma_state_estimation"
) / "mcmc_benchmark"
FIGURES_DIRECTORY = NOTEBOOK_DIRECTORY / "figures"
TABLES_DIRECTORY = NOTEBOOK_DIRECTORY / "tables"
DATA_DIRECTORY = PACKAGE_DIRECTORY.parents[1] / "data"
PARQUET = DATA_DIRECTORY / f"{dataset_utils.DEFAULT_DATASET}.parquet"
SETTINGS = DATA_DIRECTORY / f"{dataset_utils.DEFAULT_DATASET}_settings.json"

ARMS = ("map_laplace", "mcmc_nuts")
N_PARAMETERS = len(profile_parametrization_types.TWO_POWER_ZERO_EDGE_PARAM_NAMES)
PROFILE_SITES = tuple(
    f"{prefix}_{index}" for prefix in ("ne", "te") for index in range(N_PARAMETERS)
)

configuration = json.loads(
    (TABLES_DIRECTORY / "mcmc_benchmark_configuration.json").read_text()
)
profiles = pd.read_csv(TABLES_DIRECTORY / "mcmc_benchmark_profiles.csv")
profiles = profiles.loc[profiles["arm"].isin(ARMS)]
summary = pd.read_csv(TABLES_DIRECTORY / "mcmc_benchmark_summary.csv")
thomson_points = pd.read_csv(TABLES_DIRECTORY / "mcmc_benchmark_thomson_points.csv")

# The runs the tables were written from, one per truth and method.
runs = [run for run in configuration["runs"] if run["arm"] in ARMS]
loaded = {
    (run["profile_id"], run["arm"]): inference_store_utils.load_inference(
        inference_store_utils.INFERENCE_STORE, run["inference_id"]
    )
    for run in runs
}
truths = list(dict.fromkeys(run["profile_id"] for run in runs))
print(f"{len(loaded)} runs over {len(truths)} truths")

In [ ]:
record = loaded[(truths[0], "map_laplace")][0]
placement = inference_store_utils.load_placement(
    inference_store_utils.PLACEMENT_STORE, record.diagnostics_placement_id
)
seeds = sorted({run["noise_seed"] for run in runs})
nuts = configuration["nuts"]
pd.Series(
    {
        "ground truths": ", ".join(truths),
        "noise draws per truth": f"1 (seeds {seeds[0]} to {seeds[-1]})",
        "Thomson scattering volumes": len(placement.thomson_geometry.volume_positions),
        "Thomson noise model": configuration["thomson_noise_model"],
        "interferometer lines of sight": len(placement.interferometer_chords),
        "interferometer sigma_IF [m^-2]": f"{record.settings.interferometer_sigma:.3g}",
        "calibration correction per volume": (
            f"log-uniform on [1/N, N], N = {record.settings.thomson_calibration_bound}"
        ),
        "profile family": "two-power, center * (1 - rho^alpha)^beta",
        "MAP starts": f"{record.settings.map.n_starts}, drawn from the priors",
        "NUTS": (
            f"{nuts['n_chains']} chains, {nuts['n_warmup']} warmup + "
            f"{nuts['n_samples']} draws each"
        ),
        "draws per band": record.settings.n_posterior_samples,
    },
    name="setting",
).to_frame()

The prior band (every profile parameter drawn from its box), with the seven truths
over it.

In [ ]:
first_arm = profiles.loc[profiles["arm"] == ARMS[0]]
rho = first_arm.loc[first_arm["profile_id"] == truths[0], "rho"].to_numpy()
prior_density, prior_temperature = prior_study.prior_profile_curves(
    record.settings,
    jnp.asarray(rho),
    prior_study.DEFAULT_N_PRIOR_SAMPLES,
    prior_study.PRIOR_SEED,
)
prior_rows = pd.concat(
    [
        pd.DataFrame(
            {
                "profile_id": truth,
                "inference_case": "prior",
                "rho": rho,
                "ne_truth": first_arm.loc[
                    first_arm["profile_id"] == truth, "ne_truth"
                ].to_numpy(),
                "Te_truth": first_arm.loc[
                    first_arm["profile_id"] == truth, "Te_truth"
                ].to_numpy(),
                **prior_study.profile_statistics("ne", prior_density),
                **prior_study.profile_statistics("Te", prior_temperature),
            }
        )
        for truth in truths
    ],
    ignore_index=True,
)
figure = plots.plot_prior_bands(prior_rows)
pubfig.save(figure, str(FIGURES_DIRECTORY / "mcmc_benchmark_prior"))
display(figure)
plt.close(figure)

## 1. Did the MAP converge?

L-BFGS from one prior draw per run; a MAP is converged when it stops before the cap of
1000 iterations. The figure shows each profile parameter divided by its true value over
the iterations, as the mean and 5 to 95% quantiles over the truths (Fig. 3 of Merlo et
al. 2023, JINST 18 P11012). Curves that flatten before the end mean converged MAPs; quantiles that stay
away from one are truths whose MAP lies away from the truth. Iteration 0 is the point
after the first step.

In [ ]:
map_table = pd.DataFrame([run for run in runs if run["arm"] == "map_laplace"])
map_table = map_table[
    [
        "profile_id",
        "n_map_starts_converged",
        "n_map_modes",
        "map_gradient_inf_norm",
        "map_time_seconds",
    ]
].set_axis(
    ["truth", "converged starts", "distinct minima", "|grad U|_inf at MAP", "time [s]"],
    axis=1,
)
display(map_table.round({"|grad U|_inf at MAP": 4, "time [s]": 0}))

# One start per truth, so the quantiles run over the truths, as over the trials of
# Merlo et al. (2023), JINST 18 P11012.
figure = plots.plot_map_convergence(
    [
        (
            *loaded[(truth, "map_laplace")],
            desc_truth_types.read_desc_truth(truth, PARQUET, SETTINGS).profiles,
        )
        for truth in truths
    ],
    label=f"{len(truths)} truths",
)
pubfig.save(figure, str(FIGURES_DIRECTORY / "mcmc_benchmark_convergence"))
display(figure)
plt.close(figure)

## 2. Did NUTS converge?

Each chain starts at its own draw of the Laplace Gaussian widened three times, outside
the bulk of the posterior, so chains that agree at the end found the posterior.
Converged means `r_hat < 1.01` and bulk and tail ESS above 400 on every parameter
(Vehtari et al. 2021), no divergences, and at most 1% of iterations at the maximum tree
depth.

The traces show the six profile parameters and the one with the worst `r_hat`, the
warmup shaded, each beside a zoom on its first iterations. Chains that overlap and keep
fluctuating have mixed. Read section 3 only for truths that pass.

In [ ]:
columns = [
    "r_hat_max",
    "ess_bulk_min",
    "ess_tail_min",
    "n_divergences",
    "max_tree_depth_fraction",
    "nuts_converged",
]
nuts_table = summary[["profile_id"] + [f"{name}_mcmc_nuts" for name in columns]]
nuts_table = nuts_table.set_axis(
    [
        "truth",
        "max r_hat",
        "min bulk ESS",
        "min tail ESS",
        "divergences",
        "tree depth cap",
        "converged",
    ],
    axis=1,
)
display(nuts_table.round({"max r_hat": 3, "min bulk ESS": 0, "min tail ESS": 0}))

pairs_per_row = 2
zoom_iterations = 50
for truth in truths:
    record, arrays = loaded[(truth, "mcmc_nuts")]
    site_names = [str(name) for name in arrays["mcmc_site_names"]]
    worst_site = site_names[int(np.nanargmax(arrays["mcmc_r_hat"]))]
    sites = PROFILE_SITES + (() if worst_site in PROFILE_SITES else (worst_site,))
    warmup_samples = arrays["mcmc_warmup_chain_samples"]
    n_warmup = warmup_samples.shape[1]
    traces = np.concatenate([warmup_samples, arrays["mcmc_chain_samples"]], axis=1)
    n_zoom = min(zoom_iterations, n_warmup)
    n_rows = -(-len(sites) // pairs_per_row)
    figure, axes_grid = pubfig.subplots(
        n_rows,
        2 * pairs_per_row,
        width="double",
        aspect=0.86,
        squeeze=False,
        gridspec_kw={"width_ratios": (2.2, 1.0) * pairs_per_row},
    )
    for index, site in enumerate(sites):
        row, pair = divmod(index, pairs_per_row)
        axes, zoom = axes_grid[row, 2 * pair], axes_grid[row, 2 * pair + 1]
        axes.axvspan(0, n_warmup, color="0.9", lw=0.0, label="warmup", zorder=0)
        # The zoom is all warmup, so it carries the warmup shading as its background.
        zoom.set_facecolor("0.9")
        for chain, values in enumerate(traces[:, :, site_names.index(site)]):
            color = pubfig.PALETTE[chain % len(pubfig.PALETTE)]
            axes.plot(
                values, lw=0.5, alpha=0.8, color=color, label=f"chain {chain + 1}"
            )
            zoom.plot(values[:n_zoom], lw=0.8, color=color)
        axes.set_title(site.replace("_", " "), fontsize="small")
        zoom.set_title(f"first {n_zoom}", fontsize="small")
        axes.set_ylabel("parameter value")
        # Only the lowest panel of each column is labelled.
        if index + pairs_per_row >= len(sites):
            axes.set_xlabel("iteration")
            zoom.set_xlabel("iteration")
    for index in range(len(sites), n_rows * pairs_per_row):
        row, pair = divmod(index, pairs_per_row)
        axes_grid[row, 2 * pair].axis("off")
        axes_grid[row, 2 * pair + 1].axis("off")
    handles, labels = axes_grid[0, 0].get_legend_handles_labels()
    figure.legend(handles, labels, loc="outside lower center", ncols=len(labels))
    name = f"mcmc_benchmark_traces_mcmc_nuts_{truth}"
    pubfig.save(figure, str(FIGURES_DIRECTORY / name))
    display(figure)
    plt.close(figure)

## 3. Posterior comparison

The Laplace (blue, dashed) and NUTS (orange, dashed) 5 to 95% bands. Bands that coincide
mean the Gaussian approximation is adequate; where the truth leaves both, the miss
belongs to the data. The red points are each scattering volume's own `ne` and `Te`,
inverted from its five channels alone.

In [ ]:
for truth in truths:
    figure = plots.plot_method_band_comparison(
        profiles.loc[profiles["profile_id"] == truth],
        ARMS,
        thomson_points=thomson_points.loc[thomson_points["profile_id"] == truth],
    )
    pubfig.save(figure, str(FIGURES_DIRECTORY / f"mcmc_benchmark_profiles_{truth}"))
    display(figure)
    plt.close(figure)

## 4. Summary

Per truth and method, for `ne` and `Te`: the core error at `rho = 0` (% of the
true value), the RMSE over the radius (% of the true peak) and the median band half-width
(% of the local value). The last two rows are the mean and the standard deviation over
the truths.

In [ ]:
rows = []
for (truth, arm), (record, _) in loaded.items():
    accuracy = {
        item.name: item for item in inference_metrics_utils.profile_accuracy(record)
    }
    for symbol, name in (("ne", "electron_density"), ("Te", "electron_temperature")):
        rows.append(
            {
                "truth": truth,
                "profile": symbol,
                "method": plots.ARM_LABELS[arm],
                "core error [%]": accuracy[name].axis_error_percent,
                "RMSE [%]": accuracy[name].error_percent,
                "band half-width [%]": accuracy[name].uncertainty_percent,
            }
        )
metrics = pd.DataFrame(rows).pivot(index="truth", columns=["profile", "method"])
metrics = metrics.reorder_levels([1, 0, 2], axis=1)[
    pd.MultiIndex.from_product(
        [
            ("ne", "Te"),
            ("core error [%]", "RMSE [%]", "band half-width [%]"),
            [plots.ARM_LABELS[arm] for arm in ARMS],
        ]
    )
]
metrics = pd.concat([metrics, metrics.agg(["mean", "std"])])
metrics.round(2)